# Scaffold Experiment Walkthrough

This notebook runs a small synthetic Aegis experiment with an inline config and an explicit model registry. It is scaffold evidence only: it demonstrates mechanics and artifact shape, not validated trading methodology, empirical edge, or investment advice.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from tempfile import TemporaryDirectory

def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists() and (path / 'research').exists():
            return path
    raise RuntimeError('Run this notebook from inside the aegis-rd repository')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from research.aegis_research.config import resolve_experiment_config
from research.aegis_research.experiments import run_experiment
from research.aegis_research.model_plugins import make_default_model_registry


## Inline Config

The config is embedded here instead of loaded from `research/configs/experiments/`. The synthetic data, fixed FIXLB target, uncalibrated probabilities, fixed thresholds, execution assumptions, portfolio sizing, and report gates are all scaffold choices for learning the pipeline.

In [ ]:
SCAFFOLD_CONFIG = {
    'schema_version': 2,
    'name': 'synthetic_scaffold_notebook',
    'data': {
        'source': 'synthetic',
        'symbols': ['SYN'],
        'start': '2020-01-01',
        'timeframe': '1D',
        'rows': 240,
        'seed': 42,
    },
    'indicators': {
        'invalid_value_policy': 'drop_rows',
        'specs': [
            {
                'id': 'returns',
                'params': {'window': [1, 5, 20]},
                'outputs': ['returns'],
                'model_features': [{'output': 'returns'}],
            },
            {
                'id': 'ma',
                'params': {'window': [10, 30], 'wtype': 'simple'},
                'outputs': ['ma'],
                'model_features': [{'output': 'ma', 'transform': 'distance_to_close'}],
            },
            {
                'id': 'volatility',
                'params': {'window': [20]},
                'outputs': ['volatility'],
                'model_features': [{'output': 'volatility'}],
            },
            {
                'id': 'rsi',
                'params': {'window': [14], 'wtype': 'wilder'},
                'outputs': ['rsi'],
                'model_features': [{'output': 'rsi', 'transform': 'scale_0_1'}],
            },
        ],
    },
    'labels': {
        'generator': {'kind': 'fixlb', 'params': {'n': 5}},
        'target': {
            'role': 'supervised_target',
            'source_output': 'labels',
            'select': {'params': {'n': 5}},
            'transform': {
                'name': 'threshold_future_return',
                'version': 1,
                'params': {'threshold': 0.0},
            },
        },
    },
    'split': {
        'kind': 'purged_kfold',
        'n_folds': 3,
        'n_test_folds': 1,
        'purge_td': '0D',
        'embargo_td': '0D',
        'max_splits': 3,
        'max_estimated_output_cells': 500000,
        'max_public_artifact_bytes': 5000000,
    },
    'model': {
        'plugin_id': 'aegis.sklearn_logistic',
        'min_train_samples': 50,
        'params': {'max_iter': 1000, 'random_state': 42},
    },
    'signals': {
        'policy': 'long_only_hysteresis',
        'long_entry_threshold': 0.55,
        'long_exit_threshold': 0.50,
        'execution_timing': 'next_open',
    },
    'portfolio': {
        'init_cash': 10000.0,
        'fees': 0.001,
        'slippage': 0.0005,
        'entry_budget': 1.0,
        'direction': 'longonly',
    },
    'report': {
        'freq': '1D',
        'year_freq': '252D',
        'min_oos_sharpe': 0.5,
        'max_oos_drawdown': 0.35,
        'min_oos_trades': 1,
    },
}


## Run With An Explicit Registry

The registry is built in Python before config resolution. Do not put import paths, estimator definitions, API keys, provider tokens, or credentials in YAML or notebooks; use environment-backed secret references for real provider credentials.

In [ ]:
scratch = TemporaryDirectory(prefix='aegis-scaffold-')
registry = make_default_model_registry()
resolved = resolve_experiment_config(
    {**SCAFFOLD_CONFIG, 'output_dir': scratch.name},
    model_registry=registry,
)
result = run_experiment(resolved)
report = result['report']
report['status']


## Inspect Mechanics-Only Output

The status, gates, and metrics below show how the scaffold reports evidence. They are mechanics-only output from synthetic data and should not be read as a passing strategy, empirical edge, or investment advice.

In [ ]:
{
    'status': report['status'],
    'reasons': report['reasons'],
    'validation': report['validation'],
}


In [ ]:
scratch.cleanup()
